# Baseline: минимальный пайплайн детекции дефектов

Берём данные **как есть** (без EDA и очистки), конвертируем в YOLO, обучаем, оцениваем.

**Порядок работы:**
1. Загрузка данных
2. ~~EDA~~ — пропускаем
3. ~~Очистка~~ — пропускаем
4. Конвертация в YOLO
5. Обучение модели
6. Оценка: mAP и FPS

## Импорты и настройки

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import shutil
import yaml
import time
import zipfile
import torch
import matplotlib.pyplot as plt
from ultralytics import YOLO

DATA_DIR = Path(r"C:\code\neto")
TRAIN_IMAGES_DIR = DATA_DIR / "train_images"
YOLO_DIR = DATA_DIR / "yolo_dataset"
IMG_W, IMG_H = 1600, 256

## Готовые функции

Ячейку ниже нужно запустить один раз — она определяет все утилиты.

In [ ]:
def convert_to_yolo(df, image_dir, output_dir, img_w=1600, img_h=256):
    """Конвертирует DataFrame с bbox в структуру YOLO.

    Ожидает колонки: ImageId, ClassId, x_min, y_min, x_max, y_max, split.
    """
    if output_dir.exists():
        shutil.rmtree(output_dir)
    for split in ("train", "val"):
        (output_dir / "images" / split).mkdir(parents=True)
        (output_dir / "labels" / split).mkdir(parents=True)
    unique_classes = sorted(df["ClassId"].unique())
    class_map = {cls: i for i, cls in enumerate(unique_classes)}
    class_names = [f"defect_{cls}" for cls in unique_classes]
    copied, written = 0, 0
    for split in ("train", "val"):
        subset = df[df["split"] == split]
        for img_id, group in subset.groupby("ImageId"):
            src = image_dir / img_id
            dst = output_dir / "images" / split / img_id
            if src.exists() and not dst.exists():
                shutil.copy2(src, dst)
                copied += 1
            lines = []
            for _, row in group.iterrows():
                xc = ((row["x_min"] + row["x_max"]) / 2) / img_w
                yc = ((row["y_min"] + row["y_max"]) / 2) / img_h
                bw = (row["x_max"] - row["x_min"]) / img_w
                bh = (row["y_max"] - row["y_min"]) / img_h
                xc, yc = np.clip(xc, 0, 1), np.clip(yc, 0, 1)
                bw, bh = np.clip(bw, 0, 1), np.clip(bh, 0, 1)
                lines.append(f"{class_map[row['ClassId']]} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")
            label_path = output_dir / "labels" / split / (Path(img_id).stem + ".txt")
            label_path.write_text("\n".join(lines), encoding="utf-8")
            written += 1
    data_yaml = {
        "path": str(output_dir.resolve()), "train": "images/train",
        "val": "images/val", "nc": len(class_names), "names": class_names,
    }
    with open(output_dir / "data.yaml", "w", encoding="utf-8") as f:
        yaml.dump(data_yaml, f, default_flow_style=False, allow_unicode=True)
    print(f"Классы ({len(class_names)}): {class_names}")
    print(f"Скопировано изображений: {copied}, label-файлов: {written}")


def evaluate_model(model_path, data_yaml, img_dir=None, warmup=20):
    """Запускает валидацию модели (mAP) и замеряет FPS."""
    model = YOLO(model_path)
    print("=" * 50)
    print("Валидация (mAP)")
    print("=" * 50)
    metrics = model.val(data=data_yaml, verbose=False)
    print(f"mAP@0.5:      {metrics.box.map50:.4f}")
    print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
    print(f"Per-class AP@0.5: {dict(zip(metrics.names.values(), [round(v, 4) for v in metrics.box.ap50]))}")
    if img_dir is None:
        yaml_cfg = yaml.safe_load(Path(data_yaml).read_text(encoding="utf-8"))
        img_dir = Path(yaml_cfg["path"]) / yaml_cfg["val"]
    img_dir = Path(img_dir)
    images = sorted(img_dir.glob("*.jpg"))
    if not images:
        print(f"Нет изображений в {img_dir}")
        return
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"\n{'=' * 50}")
    print(f"Замер FPS ({device}, batch=1, {len(images)} изображений, warmup={warmup})")
    print("=" * 50)
    for i in range(warmup):
        model.predict(str(images[i % len(images)]), verbose=False, device=device)
    times = []
    for img_path in images:
        t0 = time.perf_counter()
        model.predict(str(img_path), verbose=False, device=device)
        times.append(time.perf_counter() - t0)
    times = np.array(times)
    print(f"Среднее время на кадр: {times.mean()*1000:.1f} ms")
    print(f"FPS:                   {1/times.mean():.1f}")
    print(f"Требование 30 FPS:     {'PASS' if 1/times.mean() >= 30 else 'FAIL'}")


def export_for_cvat(yolo_dir, split="val", output_zip=None):
    """Собирает ZIP-датасет для импорта в CVAT (формат YOLO 1.1).

    Архив содержит изображения и аннотации — готов для
    CVAT → Create task → Import dataset → YOLO 1.1.
    """
    yolo_dir = Path(yolo_dir)
    labels_dir = yolo_dir / "labels" / split
    images_dir = yolo_dir / "images" / split
    with open(yolo_dir / "data.yaml", encoding="utf-8") as f:
        data_cfg = yaml.safe_load(f)
    class_names = data_cfg["names"]
    if output_zip is None:
        output_zip = yolo_dir.parent / f"cvat_{split}.zip"
    label_files = sorted(labels_dir.glob("*.txt"))
    with zipfile.ZipFile(output_zip, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.writestr("obj.names", "\n".join(class_names))
        obj_data = f"classes = {len(class_names)}\nnames = obj.names\ntrain = train.txt\nbackup = backup/\n"
        zf.writestr("obj.data", obj_data)
        train_txt = []
        for lf in label_files:
            zf.write(lf, f"obj_train_data/{lf.name}")
            img_path = images_dir / (lf.stem + ".jpg")
            if img_path.exists():
                zf.write(img_path, f"obj_train_data/{img_path.name}")
            train_txt.append(f"obj_train_data/{lf.stem}.jpg")
        zf.writestr("train.txt", "\n".join(train_txt))
    print(f"ZIP для CVAT: {output_zip} ({len(label_files)} файлов)")
    return output_zip

print("Все функции загружены.")

---
## Шаг 1. Загрузка данных

In [ ]:
df = pd.read_csv(DATA_DIR / "train_bboxes.csv")
print(f"Аннотаций: {len(df)}, изображений: {df['ImageId'].nunique()}")
print(f"Классы: {sorted(df['ClassId'].unique())}")
print(df["ClassId"].value_counts().sort_index())
print(f"\nСплит: {df['split'].value_counts().to_dict()}")
df.head()

---
## Шаг 2. EDA

В baseline пропускаем — берём данные как есть.

---
## Шаг 3. Очистка данных

В baseline пропускаем — никаких преобразований.

---
## Шаг 4. Конвертация в YOLO

In [ ]:
print("Распределение классов по сплитам:")
print(df.groupby(["split", "ClassId"]).size().unstack(fill_value=0))

In [ ]:
convert_to_yolo(df, image_dir=TRAIN_IMAGES_DIR, output_dir=YOLO_DIR)

---
## Шаг 5. Обучение модели

In [ ]:
model = YOLO("yolo11n.pt")  # nano — самая быстрая
results = model.train(
    data=str(YOLO_DIR / "data.yaml"),
    epochs=50,
    imgsz=256,
    batch=16,
    project=str(DATA_DIR / "runs"),
    name="baseline",
)

---
## Шаг 6. Оценка качества

Функция `evaluate_model` выведет mAP@0.5 и замерит FPS.
Минимальное требование: **30 FPS** на GPU.

In [ ]:
evaluate_model(
    model_path=str(DATA_DIR / "runs" / "baseline" / "weights" / "best.pt"),
    data_yaml=str(YOLO_DIR / "data.yaml"),
)

---
## Экспорт для CVAT (опционально)

Если хотите визуально проверить аннотации в CVAT:
1. Запустите ячейку ниже — она создаст ZIP
2. В CVAT: создайте задачу, загрузите изображения
3. Menu → Upload annotations → YOLO 1.1 → выберите ZIP

In [ ]:
# export_for_cvat(YOLO_DIR, split="val")